In [1]:
import pandas as pd

df_educacion = pd.read_csv("../datasets/noticias_educacion_sample.csv")
df_educacion['clase'] = 0
df_politica = pd.read_csv("../datasets/noticias_politica_sample.csv")
df_politica['clase'] = 1
df_deportes = pd.read_csv("../datasets/noticias_deportes_sample.csv")
df_deportes['clase'] = 2
df_economia = pd.read_csv("../datasets/noticias_economia_sample.csv")
df_economia['clase'] = 3

In [2]:
df = pd.concat([df_educacion, df_politica, df_deportes, df_economia]).dropna().reset_index()
df

,index,content,date,headline,description,clase
0,0,Como parte de la política de puertas abiertas ...,2022-02-08T19:12:01.737Z,La CAN abre convocatorias para pasantías en Co...,La Comunidad Andina de Naciones abrió la posib...,0
1,1,"El programa, que cumple 30 años desde su prime...",2022-05-14T18:02:23.629Z,Colfuturo apoyará a 1.526 profesionales colomb...,"Los beneficiarios, en su mayoría, realizaron e...",0
2,2,Estudiar una carrera universitaria en Colombia...,2022-10-19T09:45:01.712Z,¿Cómo estudiar becado en la mejor universidad ...,"Según el ranking de Times Higher Education, la...",0
3,3,Escuche aquí el episodio número 27 de Finanzas...,2021-04-07T17:56:34.238Z,Consejos para financiar con inteligencia sus e...,Si estudiar es uno de sus principales objetivo...,0
4,4,Durante el último año de la carrera universita...,2022-04-02T18:08:22.865Z,Pruebas Saber Pro: el listado de universidades...,Las universidades públicas presentaron preocup...,0
...,...,...,...,...,...,...
1928,495,Colombia sigue aumentando su endeudamiento ext...,2023-02-10T23:08:47.922Z,"Deuda externa de Colombia representó el 52,8% ...",Así lo deja en evidencia el más reciente repor...,3
1929,496,La Agencia de Estados Unidos para el Desarroll...,2022-09-28T17:00:15.603Z,Lanzan convocatoria para apoyar a más de mil o...,La Usaid estará al frente de este proceso que ...,3
1930,497,La inflación es uno de los mayores retos que e...,2023-02-25T03:41:20.639Z,Controlar la inflación no será tan fácil como ...,El aumento en los precios será una constante e...,3
1931,498,23 lugares icónicos de Cúcuta fueron decorados...,2022-12-07T17:16:46.317Z,Reapertura económica en la frontera: artesanas...,Cúcuta prepara la Ruta Navideña luego de haber...,3


# BERT:

In [3]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

# 1. Preparar el dataset (las 4 clases)
df_clf = df[['content', 'clase']].dropna().reset_index(drop=True)

# 2. Split train / val / test
train_df, temp_df = train_test_split(df_clf, test_size=0.3, stratify=df_clf['clase'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['clase'], random_state=42)

# 3. Tokenizador BERT
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

def tokenize(examples):
    return tokenizer(examples["content"], padding="max_length", truncation=True, max_length=128)

# 4. Convertir a HuggingFace Dataset
def to_hf_dataset(df_split):
    ds = Dataset.from_pandas(
        df_split[['content', 'clase']].rename(columns={'clase': 'labels'}).reset_index(drop=True)
    )
    ds = ds.map(tokenize, batched=True, remove_columns=['content'])
    return ds

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)

# 5. Cargar modelo BERT pre-entrenado para clasificación (4 clases)
# Uso multilingual-cased porque el texto está en español
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=4
)

# 6. Configuración de entrenamiento
training_args = TrainingArguments(
    output_dir="./bert-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=5e-5,
    logging_steps=50,
    report_to="none",
)

# 7. Trainer (HuggingFace maneja el loop de entrenamiento, optimizer, etc.)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

# 8. Entrenar
trainer.train()

# 9. Evaluar en test set
predictions = trainer.predict(test_ds)
y_test = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

# 10. Métricas por clase
print()
class_names = ['Educación', 'Política', 'Deportes', 'Economía']
for i in range(4):
    class_predicted = [1 if x == i else 0 for x in y_pred]
    class_real = [1 if x == i else 0 for x in y_test]
    p = precision_score(class_real, class_predicted, zero_division=0)
    r = recall_score(class_real, class_predicted, zero_division=0)
    f1 = f1_score(class_real, class_predicted, zero_division=0)
    print(f"Clase {i} ({class_names[i]}): P={p:.2f}  R={r:.2f}  F1={f1:.2f}")

# 11. Predicción sobre una oración nueva
test_text = "La universidad abrió una nueva convocatoria para becas internacionales."
inputs = tokenizer(test_text, return_tensors="pt", truncation=True, padding=True, max_length=128)
inputs = {k: v.to(model.device) for k, v in inputs.items()}
with torch.no_grad():
    outputs = model(**inputs)
probs = torch.softmax(outputs.logits, dim=-1)
predicted_class = torch.argmax(probs, dim=1).item()
print(f"\nPredicción para texto de ejemplo: clase {predicted_class} ({class_names[predicted_class]})")

/Users/davidzarruk/Documents/MIAD_ML_NLP_2026/.env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Map:   0%|          | 0/1353 [00:00<?, ? examples/s]

Map:  74%|███████▍  | 1000/1353 [00:00<00:00, 4086.56 examples/s]

Map: 100%|██████████| 1353/1353 [00:00<00:00, 4110.21 examples/s]

Map:   0%|          | 0/290 [00:00<?, ? examples/s]

...

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11166.70it/s]


BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


/Users/davidzarruk/Documents/MIAD_ML_NLP_2026/.env/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.788922,0.297537
2,0.227216,0.213836
3,0.081340,0.225362



Clase 0 (Educación): P=0.94  R=0.92  F1=0.93
Clase 1 (Política): P=0.91  R=0.85  F1=0.88
Clase 2 (Deportes): P=0.99  R=1.00  F1=0.99
Clase 3 (Economía): P=0.80  R=0.87  F1=0.83

Predicción para texto de ejemplo: clase 0 (Educación)


# GPT-2:

In [4]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)
import torch

# 1. Cargar dataset
df_gpt = df[['content']].dropna().reset_index(drop=True)
dataset = Dataset.from_pandas(df_gpt)

# 2. Tokenizador
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples["content"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])

# 3. Cargar GPT-2 pre-entrenado
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))

# 4. Data collator para causal LM
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# 5. Configuración de entrenamiento
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_strategy="no",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

# 6. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# 7. Entrenar
trainer.train()

# 8. Función de generación
def generate_text(prompt, max_length=100, temperature=0.9):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        input_ids,
        max_length=max_length,
        do_sample=True,
        temperature=temperature,
        top_k=50,
        top_p=0.95,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id,
        attention_mask=torch.ones_like(input_ids),
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Ejemplo de generación
print(generate_text("Como parte de la política"))

Map:   0%|          | 0/1933 [00:00<?, ? examples/s]

Map:  52%|█████▏    | 1000/1933 [00:00<00:00, 2650.77 examples/s]

Map: 100%|██████████| 1933/1933 [00:00<00:00, 3022.47 examples/s]

Map: 100%|██████████| 1933/1933 [00:00<00:00, 2924.96 examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 9290.54it/s]

/Users/davidzarruk/Documents/MIAD_ML_NLP_2026/.env/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,4.320991
200,3.937763
300,3.812304
400,3.682724
500,3.597662
600,3.405990
700,3.378541
800,3.354366
900,3.324688
1000,3.291125


Como parte de la política que el director al trabajo que la plataforma que es la líderes tener en Colombia hablar asegún la reforma. Se registrado aplicaron el secto de la economía de Bogotá y los partidos en la salida en el que el presidente y el gobierno entrevista. El mandato con la mandatario al cese


In [5]:
print(generate_text("Gustavo Petro y el Pacto"))

Gustavo Petro y el Pacto Histórico se bien comenzó el Gobierno nacional de Colombia contra el presidente Gustavo Petro, y un gran técnico para el Gobierno, se había que llegará con el tiempo de las gremios de ‘aúle’, ya está pidió más de la región por la que se han mín de un
